In [ ]:
#!/usr/bin/env python3
import os
import sys
import time
import json
import requests
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

API_KEY = ""

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

CSV_PATH = "/content/BSMDD_v3_2000_stratified.csv"

MODEL = "qwen/qwen3.7-plus" #change

SYSTEM_PROMPT = """Classify the given Bangla social media text into following categories:
1 = Depressive (hopelessness, suicidal, self-harm, clear depression).
0 = Non-depressive (stress/anger alone doesn't equal depression).
Output EXACTLY '0' or '1'. No explanations. Nothing else. only "0" and "1". """

USER_TEMPLATE = "Text: {text}\nLabel:"

MAX_RETRIES = 5
BASE_BACKOFF = 2.0
REQUEST_TIMEOUT = 30

SITE_URL = "https://colab.research.google.com"
SITE_NAME = "Bangla Benchmark"


def build_headers():
    if not API_KEY:
        raise RuntimeError("OPENROUTER_API_KEY not set in colab secrets")
    return {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": SITE_URL,
        "X-Title": SITE_NAME,
    }


def parse_label(raw_text):
    if raw_text is None:
        return None
    s = raw_text.strip()
    for ch in s:
        if ch == "0":
            return 0
        if ch == "1":
            return 1
    return None


def call_model(model, text, headers):
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_TEMPLATE.format(text=text)},
        ],
        "max_tokens": 5,
        "temperature": 0,
        "reasoning": {
          "effort": "none"
        },
    }

    start = time.monotonic()

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=REQUEST_TIMEOUT)
        except requests.RequestException as e:
            if attempt == MAX_RETRIES:
                return None, f"request_error:{e}", time.monotonic() - start
            time.sleep(BASE_BACKOFF * attempt)
            continue

        if resp.status_code == 429:
            retry_after = resp.headers.get("Retry-After")
            wait = float(retry_after) if retry_after else BASE_BACKOFF * attempt
            time.sleep(wait)
            continue

        if resp.status_code >= 500:
            time.sleep(BASE_BACKOFF * attempt)
            continue

        if resp.status_code != 200:
            return None, f"http_error:{resp.status_code}:{resp.text[:200]}", time.monotonic() - start

        try:
            data = resp.json()
            content = data["choices"][0]["message"]["content"]
        except (KeyError, IndexError, json.JSONDecodeError) as e:
            return None, f"parse_error:{e}", time.monotonic() - start

        return content, None, time.monotonic() - start

    return None, "max_retries_exceeded", time.monotonic() - start


def checkpoint_path(out_dir, model):
    col_name = model.replace("/", "__")
    return os.path.join(out_dir, f"checkpoint_{col_name}.csv")


def load_checkpoint(out_dir, model):
    path = checkpoint_path(out_dir, model)
    if os.path.exists(path):
        return pd.read_csv(path)
    return None


def save_checkpoint(out_dir, model, rows):
    path = checkpoint_path(out_dir, model)
    pd.DataFrame(rows).to_csv(path, index=False)


def run_model_on_dataset(model, df, headers, out_dir, checkpoint_every=10):
    existing = load_checkpoint(out_dir, model)
    rows = existing.to_dict("records") if existing is not None else []
    start_idx = len(rows)

    if start_idx >= len(df):
        return rows

    texts = df["text"].tolist()
    labels = df["label"].tolist()

    pbar = tqdm(range(start_idx, len(df)), desc=model, unit="row", initial=start_idx, total=len(df))
    for i in pbar:
        text = texts[i]
        label = labels[i]
        raw, err, latency = call_model(model, text, headers)
        pred = parse_label(raw)
        # print("\n")
        # print(f"[{i}] raw='{raw}' err='{err}' → pred={pred} | label={label}") #add
        # print("\n")


        if pred is None:
          raw, err, latency = call_model(model, text, headers)
          pred = parse_label(raw)
          print("After Retrying: ")
          # print("\n")
          # print(f"[{i}] raw='{raw}' err='{err}' → pred={pred} | label={label}")
          # print("\n")





        correct = (pred == label) if pred is not None else False

        rows.append({
            "text": text,
            "label": label,
            "prediction": pred,
            "correct": correct,
            "latency": latency,
            "raw_response": raw,
            "error": err,
        })

        if (i - start_idx + 1) % checkpoint_every == 0:
            save_checkpoint(out_dir, model, rows)

    save_checkpoint(out_dir, model, rows)
    return rows


def compute_metrics(rows):
    valid = [r for r in rows if r["prediction"] is not None]
    if not valid:
        return {
            "accuracy": None, "precision": None, "recall": None,
            "f1": None, "confusion_matrix": None,
            "valid_predictions": 0, "total": len(rows),
            "avg_latency": None,
        }

    yt = [r["label"] for r in valid]
    yp = [r["prediction"] for r in valid]
    latencies = [r["latency"] for r in rows if r["latency"] is not None]

    acc = accuracy_score(yt, yp)
    prec = precision_score(yt, yp, zero_division=0)
    rec = recall_score(yt, yp, zero_division=0)
    f1 = f1_score(yt, yp, zero_division=0)
    cm = confusion_matrix(yt, yp, labels=[0, 1]).tolist()

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "confusion_matrix": cm,
        "valid_predictions": len(valid),
        "total": len(rows),
        "avg_latency": sum(latencies) / len(latencies) if latencies else None,
    }


def main():
    # Configuration
    NUM_SAMPLES = None   # Change
    OUTPUT_DIR = "/content"
    CHECKPOINT_EVERY = 20

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    df = pd.read_csv(CSV_PATH)

    if "text" not in df.columns or "label" not in df.columns:
        raise ValueError("CSV must contain 'text' and 'label' columns")

    df["label"] = df["label"].astype(int)

    if NUM_SAMPLES is not None:
        df = df.head(NUM_SAMPLES).reset_index(drop=True)

    headers = build_headers()

    rows = run_model_on_dataset(
        MODEL,
        df,
        headers,
        OUTPUT_DIR,
        CHECKPOINT_EVERY
    )

    pred_rows = []
    for r in rows:
        pred_rows.append({
            "text": r["text"],
            "label": r["label"],
            "prediction": r["prediction"],
            "correct": r["correct"],
            "latency": r["latency"],
            "raw_response": r["raw_response"],
            "error": r.get("error"),
        })

    metrics = compute_metrics(rows)

    results_row = {
        "model": MODEL,
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "confusion_matrix": json.dumps(metrics["confusion_matrix"]),
        "valid_predictions": metrics["valid_predictions"],
        "total": metrics["total"],
        "avg_latency": metrics["avg_latency"],
    }

    pred_path = os.path.join(OUTPUT_DIR, "predictions.csv")
    pd.DataFrame(pred_rows).to_csv(pred_path, index=False, encoding="utf-8-sig")

    results_path = os.path.join(OUTPUT_DIR, "benchmark_results.csv")
    pd.DataFrame([results_row]).to_csv(
    results_path,
    index=False,
    encoding="utf-8-sig"
    )
    print(pd.DataFrame([results_row]).to_string(index=False))


if __name__ == "__main__":
    main()

qwen/qwen3.7-plus: 100%|██████████| 2000/2000 [6:22:29<00:00, 11.47s/row]

            model  accuracy  precision   recall       f1         confusion_matrix  valid_predictions  total  avg_latency
qwen/qwen3.7-plus    0.8295   0.798913 0.881119 0.838005 [[777, 222], [119, 882]]               2000   2000    11.471755
